# 13 — YOLO26s + Object-Aware Transformer

**Controlled detector comparison experiment.**

This notebook uses the **same Object-Aware Transformer architecture and training/evaluation protocol as the RT-DETR-X experiment**. Only the object-detector input is changed to **YOLO26s**.

- Dataset: Flickr8K + Flickr30K
- Images: 39,874
- Captions: 199,369
- Test images: 3,988
- References/test captions: 19,940 / 5 per image
- YOLO26s detections: existing permanent CSV; detection is **not rerun**
- Confidence threshold: 0.30
- Top-K objects: 10
- Spatial features: 9
- D_MODEL: 256
- Encoder/decoder layers: 3 / 3
- Epochs: 20
- Batch size: 64
- LR: 1e-4
- Weight decay: 1e-4
- Label smoothing: 0.1

**Important:** do not load the RT-DETR-X checkpoint. YOLO26s must be trained as a fresh model so the detector comparison remains controlled.


> **Drive-only rule:** this notebook never reads a dataset, checkpoint, or cached artifact from `/content`. It uses the permanent Google Drive files under `MyDrive/Image_captioning`. Temporary Python/Torch runtime memory is unavoidable, but no dataset cache is created in `/content`.


In [ ]:
# ============================================================================
# NOTEBOOK 13 — CELL 1
# FRESH DRIVE-ONLY SETUP
# ============================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from collections import defaultdict
import json
import math
import random
import time
import warnings
import os

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Image_captioning"
)

assert DRIVE_ROOT.exists()

# --------------------------------------------------------------------------
# Permanent input files
# --------------------------------------------------------------------------

YOLO26_CSV = (
    DRIVE_ROOT
    / "Outputs"
    / "YOLO26_Detection"
    / "Final"
    / "yolo26_detections_full.csv"
)

CAPTION_VOCAB_FILE = (
    DRIVE_ROOT
    / "Outputs"
    / "Structured_Captioning"
    / "caption_vocabulary_train_only.json"
)

CAPTION_SEQ_FILE = (
    DRIVE_ROOT
    / "Features"
    / "Structured_Object"
    / "caption_sequences_train_only.npz"
)

OBJECT_VOCAB_FILE = (
    DRIVE_ROOT
    / "Outputs"
    / "Structured_Captioning"
    / "object_vocabulary.json"
)

TRAIN_SPLIT_FILE = (
    DRIVE_ROOT
    / "Outputs"
    / "Structured_Captioning"
    / "train_image_ids.csv"
)

VAL_SPLIT_FILE = (
    DRIVE_ROOT
    / "Outputs"
    / "Structured_Captioning"
    / "val_image_ids.csv"
)

TEST_SPLIT_FILE = (
    DRIVE_ROOT
    / "Outputs"
    / "Structured_Captioning"
    / "test_image_ids.csv"
)

# --------------------------------------------------------------------------
# Permanent output locations
# --------------------------------------------------------------------------

FEATURE_DIR = (
    DRIVE_ROOT
    / "Features"
    / "YOLO26_Object_Aware_Transformer"
)

OUTPUT_DIR = (
    DRIVE_ROOT
    / "Outputs"
    / "YOLO26_Object_Aware_Transformer"
)

CHECKPOINT_DIR = OUTPUT_DIR / "Checkpoints"
EVALUATION_DIR = OUTPUT_DIR / "Evaluation"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

YOLO26_FEATURE_FILE = (
    FEATURE_DIR
    / "yolo26_structured_object_features.npz"
)

BEST_CHECKPOINT = (
    CHECKPOINT_DIR
    / "best_yolo26_object_aware_transformer.pt"
)

LAST_CHECKPOINT = (
    CHECKPOINT_DIR
    / "last_yolo26_object_aware_transformer.pt"
)

HISTORY_FILE = (
    OUTPUT_DIR
    / "yolo26_training_history.json"
)

# --------------------------------------------------------------------------
# Experiment configuration
# --------------------------------------------------------------------------

CONF_THRESHOLD = 0.30
TOP_K = 10

SPATIAL_DIM = 9
MAX_LEN = 32

D_MODEL = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3
DIM_FEEDFORWARD = 1024
DROPOUT = 0.2

BATCH_SIZE = 64
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

NUM_EPOCHS = 20
PATIENCE = 4

LR = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
GRAD_CLIP = 1.0

# --------------------------------------------------------------------------
# Verify permanent inputs
# --------------------------------------------------------------------------

required = {
    "YOLO26 CSV": YOLO26_CSV,
    "Caption vocabulary": CAPTION_VOCAB_FILE,
    "Caption sequences": CAPTION_SEQ_FILE,
    "Object vocabulary": OBJECT_VOCAB_FILE,
    "Train split": TRAIN_SPLIT_FILE,
    "Val split": VAL_SPLIT_FILE,
    "Test split": TEST_SPLIT_FILE,
}

print("=" * 75)
print("NOTEBOOK 13 — YOLO26s OBJECT-AWARE TRANSFORMER")
print("=" * 75)

print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")

for name, path in required.items():
    assert path.exists(), f"MISSING: {path}"
    print(f"✓ {name}")

print()
print("✓ Fresh runtime")
print("✓ Drive-only")
print("✓ No local dataset")
print("✓ No cache")
print("✓ YOLO26 detection will NOT be rerun")
print("=" * 75)

Mounted at /content/drive
NOTEBOOK 13 — YOLO26s OBJECT-AWARE TRANSFORMER
Device: cuda
GPU:    Tesla T4
✓ YOLO26 CSV
✓ Caption vocabulary
✓ Caption sequences
✓ Object vocabulary
✓ Train split
✓ Val split
✓ Test split

✓ Fresh runtime
✓ Drive-only
✓ No local dataset
✓ No cache
✓ YOLO26 detection will NOT be rerun


In [ ]:
# ============================================================================
# CELL 2 — LOAD CAPTION DATA + EXACT SPLITS
# ============================================================================

print("=" * 75)
print("CELL 2 — LOADING CAPTION DATA AND SPLITS")
print("=" * 75)

# ============================================================================
# CAPTION VOCABULARY
# ============================================================================

with open(
    CAPTION_VOCAB_FILE,
    "r",
    encoding="utf-8"
) as f:
    vocab_raw = json.load(f)

caption_to_id = vocab_raw.get(
    "caption_to_id",
    vocab_raw
)

caption_to_id = {
    str(k): int(v)
    for k, v in caption_to_id.items()
}

id_to_caption = {
    int(v): str(k)
    for k, v in caption_to_id.items()
}

PAD_ID = caption_to_id["<pad>"]
START_ID = caption_to_id["<start>"]
END_ID = caption_to_id["<end>"]
UNK_ID = caption_to_id["<unk>"]

assert len(caption_to_id) == 12666
assert PAD_ID == 0
assert START_ID == 1
assert END_ID == 2
assert UNK_ID == 3

# ============================================================================
# CAPTION SEQUENCES
# ============================================================================

caption_data = np.load(
    CAPTION_SEQ_FILE,
    allow_pickle=True
)

caption_image_ids = (
    caption_data["image_ids"]
    .astype(str)
)

caption_sequences = (
    caption_data["caption_ids"]
    .astype(np.int64)
)

assert caption_image_ids.shape == (199369,)
assert caption_sequences.shape == (199369, 32)

# ============================================================================
# SPLITS
# ============================================================================

def load_split(path):

    df = pd.read_csv(path)

    column = (
        "image_id"
        if "image_id" in df.columns
        else df.columns[0]
    )

    return sorted(
        df[column]
        .astype(str)
        .str.strip()
        .tolist()
    )


train_image_ids = load_split(
    TRAIN_SPLIT_FILE
)

val_image_ids = load_split(
    VAL_SPLIT_FILE
)

test_image_ids = load_split(
    TEST_SPLIT_FILE
)

assert len(train_image_ids) == 31899
assert len(val_image_ids) == 3987
assert len(test_image_ids) == 3988

train_set = set(train_image_ids)
val_set = set(val_image_ids)
test_set = set(test_image_ids)

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

# ============================================================================
# CAPTION INDEX MAP
# ============================================================================

image_to_caption_indices = defaultdict(list)

for idx, image_id in enumerate(
    caption_image_ids
):
    image_to_caption_indices[
        image_id
    ].append(idx)


def get_caption_indices(image_list):

    return np.asarray(
        [
            idx
            for image_id in image_list
            for idx in image_to_caption_indices[
                image_id
            ]
        ],
        dtype=np.int64
    )


train_caption_indices = get_caption_indices(
    train_image_ids
)

val_caption_indices = get_caption_indices(
    val_image_ids
)

test_caption_indices = get_caption_indices(
    test_image_ids
)

assert len(train_caption_indices) == 159494
assert len(val_caption_indices) == 19935
assert len(test_caption_indices) == 19940

# ============================================================================
# OBJECT VOCABULARY
# ============================================================================

with open(
    OBJECT_VOCAB_FILE,
    "r",
    encoding="utf-8"
) as f:
    object_vocab_raw = json.load(f)

object_to_id = object_vocab_raw.get(
    "object_to_id",
    object_vocab_raw
)

object_to_id = {
    str(k): int(v)
    for k, v in object_to_id.items()
}

id_to_object = {
    int(v): str(k)
    for k, v in object_to_id.items()
}

assert len(object_to_id) == 82

def norm_object_name(x):

    return (
        str(x)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

normalized_object_to_id = {
    norm_object_name(k): int(v)
    for k, v in object_to_id.items()
}

print()
print("=" * 75)
print("✓ CAPTION DATA LOADED")
print("=" * 75)

print(f"Vocabulary:       {len(caption_to_id):,}")
print(f"Caption records:  {len(caption_image_ids):,}")
print(f"Object vocabulary:{len(object_to_id):,}")

print()
print(f"Train images:     {len(train_image_ids):,}")
print(f"Val images:       {len(val_image_ids):,}")
print(f"Test images:      {len(test_image_ids):,}")

print()
print(f"Train captions:   {len(train_caption_indices):,}")
print(f"Val captions:     {len(val_caption_indices):,}")
print(f"Test captions:    {len(test_caption_indices):,}")

print()
print("✓ Exact image-disjoint splits verified")
print("=" * 75)

CELL 2 — LOADING CAPTION DATA AND SPLITS

✓ CAPTION DATA LOADED
Vocabulary:       12,666
Caption records:  199,369
Object vocabulary:82

Train images:     31,899
Val images:       3,987
Test images:      3,988

Train captions:   159,494
Val captions:     19,935
Test captions:    19,940

✓ Exact image-disjoint splits verified


In [ ]:
# ============================================================================
# CELL 3 — FAST YOLO26s STRUCTURED OBJECT FEATURES
# FIX: DUPLICATE FLICKR30K IMAGE PATHS
# DRIVE-ONLY
# ============================================================================

print("=" * 75)
print("CELL 3 — FAST YOLO26s STRUCTURED OBJECT FEATURES")
print("=" * 75)

import concurrent.futures
from PIL import Image

# ============================================================================
# READ YOLO26 DETECTIONS
# ============================================================================

print("Reading YOLO26 detection CSV...")

yolo_df = pd.read_csv(YOLO26_CSV)

print(
    f"Detection rows loaded: {len(yolo_df):,}"
)

required_cols = [
    "image_id",
    "object_name",
    "confidence",
    "xmin",
    "ymin",
    "xmax",
    "ymax"
]

missing_cols = [
    c for c in required_cols
    if c not in yolo_df.columns
]

assert not missing_cols, (
    f"Missing columns: {missing_cols}"
)

# ============================================================================
# CLEAN
# ============================================================================

yolo_df["image_id"] = (
    yolo_df["image_id"]
    .astype(str)
    .str.strip()
)

yolo_df["object_name"] = (
    yolo_df["object_name"]
    .fillna("")
    .astype(str)
    .map(norm_object_name)
)

for col in [
    "confidence",
    "xmin",
    "ymin",
    "xmax",
    "ymax"
]:

    yolo_df[col] = pd.to_numeric(
        yolo_df[col],
        errors="coerce"
    ).fillna(0.0)

yolo_df = yolo_df[
    yolo_df["object_name"].ne("")
].copy()

yolo_df = yolo_df[
    yolo_df["confidence"] >= CONF_THRESHOLD
].copy()

print(
    f"Rows after confidence >= {CONF_THRESHOLD}: "
    f"{len(yolo_df):,}"
)

# ============================================================================
# REQUIRED IMAGE FILENAMES
# ============================================================================

unique_detection_ids = (
    yolo_df["image_id"]
    .drop_duplicates()
    .tolist()
)

required_basenames = {
    Path(x).name
    for x in unique_detection_ids
}

print(
    f"Unique detection images required: "
    f"{len(required_basenames):,}"
)

# ============================================================================
# SEARCH DRIVE
# ============================================================================

print()
print("Searching permanent Google Drive images...")

F8K_DIR = (
    DRIVE_ROOT
    / "Dataset"
    / "Raw"
    / "Flickr8K"
    / "Images"
)

F30K_DIR = (
    DRIVE_ROOT
    / "Dataset"
    / "Raw"
    / "Flickr30k"
)

assert F8K_DIR.exists()
assert F30K_DIR.exists()

basename_to_paths = defaultdict(list)

search_start = time.time()

for root, dirs, files in os.walk(
    DRIVE_ROOT
):

    dirs[:] = [
        d for d in dirs
        if d != ".ipynb_checkpoints"
    ]

    for filename in files:

        if Path(filename).suffix.lower() not in {
            ".jpg",
            ".jpeg",
            ".png"
        }:
            continue

        if filename in required_basenames:

            basename_to_paths[
                filename
            ].append(
                Path(root) / filename
            )

print(
    f"Drive image search complete in "
    f"{time.time() - search_start:.1f}s"
)

print(
    f"Unique image filenames found: "
    f"{len(basename_to_paths):,}"
)

# ============================================================================
# VERIFY REQUIRED FILENAMES
# ============================================================================

missing_basenames = sorted(
    required_basenames
    - set(basename_to_paths.keys())
)

if missing_basenames:

    raise FileNotFoundError(
        f"{len(missing_basenames):,} required image files "
        "were not found on Drive.\nExamples:\n"
        + "\n".join(
            missing_basenames[:20]
        )
    )

# ============================================================================
# RESOLVE DUPLICATES
#
# Some Flickr30K files exist twice:
#
#   .../flickr30k_images/IMAGE.jpg
#   .../flickr30k_images/flickr30k_images/IMAGE.jpg
#
# Choose the SHALLOWEST path.
# ============================================================================

image_path_map = {}

duplicate_count = 0

for image_id in unique_detection_ids:

    basename = Path(
        image_id
    ).name

    candidates = (
        basename_to_paths[
            basename
        ]
    )

    # Prefer the shallowest path.
    chosen = min(
        candidates,
        key=lambda p: (
            len(p.parts),
            len(str(p))
        )
    )

    image_path_map[
        image_id
    ] = chosen

    if len(candidates) > 1:

        duplicate_count += 1

print(
    f"Duplicate filename groups: "
    f"{duplicate_count:,}"
)

print(
    "✓ Duplicate paths resolved "
    "using shallowest Drive path"
)

assert len(image_path_map) == (
    len(unique_detection_ids)
)

# ============================================================================
# READ EACH UNIQUE IMAGE DIMENSION ONCE
# ============================================================================

print()
print(
    f"Reading dimensions for "
    f"{len(image_path_map):,} unique images..."
)

def read_image_size(item):

    image_id, path = item

    try:

        with Image.open(path) as img:

            return (
                image_id,
                int(img.width),
                int(img.height),
                None
            )

    except Exception as e:

        return (
            image_id,
            None,
            None,
            str(e)
        )


size_map = {}
size_errors = []

size_start = time.time()

items = list(
    image_path_map.items()
)

with concurrent.futures.ThreadPoolExecutor(
    max_workers=16
) as executor:

    results = executor.map(
        read_image_size,
        items
    )

    for i, result in enumerate(
        results,
        start=1
    ):

        image_id, width, height, error = result

        if error is not None:

            size_errors.append(
                (
                    image_id,
                    error
                )
            )

        else:

            size_map[
                image_id
            ] = (
                width,
                height
            )

        if (
            i % 5000 == 0
            or i == len(items)
        ):

            elapsed = (
                time.time()
                - size_start
            )

            print(
                f"  {i:,}/{len(items):,} "
                f"| {i/max(elapsed,1e-9):.0f} img/s",
                flush=True
            )

if size_errors:

    print("\nImage read errors:")

    for image_id, error in size_errors[:10]:

        print(
            image_id,
            error
        )

    raise RuntimeError(
        f"Failed to read "
        f"{len(size_errors):,} images."
    )

assert len(size_map) == (
    len(image_path_map)
)

# ============================================================================
# MAP IMAGE DIMENSIONS TO DETECTIONS
# ============================================================================

width_map = {
    image_id: size[0]
    for image_id, size
    in size_map.items()
}

height_map = {
    image_id: size[1]
    for image_id, size
    in size_map.items()
}

yolo_df["_img_width"] = (
    yolo_df["image_id"]
    .map(width_map)
)

yolo_df["_img_height"] = (
    yolo_df["image_id"]
    .map(height_map)
)

assert (
    yolo_df["_img_width"]
    .notna()
    .all()
)

assert (
    yolo_df["_img_height"]
    .notna()
    .all()
)

# ============================================================================
# NORMALIZE BOUNDING BOXES
# ============================================================================

coord_max = float(
    yolo_df[
        [
            "xmin",
            "ymin",
            "xmax",
            "ymax"
        ]
    ]
    .to_numpy()
    .max()
)

print()
print(
    f"Maximum raw box coordinate: "
    f"{coord_max:.4f}"
)

if coord_max > 1.5:

    print(
        "Absolute pixel coordinates detected."
    )

    yolo_df["xmin"] /= (
        yolo_df["_img_width"]
    )

    yolo_df["xmax"] /= (
        yolo_df["_img_width"]
    )

    yolo_df["ymin"] /= (
        yolo_df["_img_height"]
    )

    yolo_df["ymax"] /= (
        yolo_df["_img_height"]
    )

else:

    print(
        "Coordinates already normalized."
    )

# ============================================================================
# CLAMP
# ============================================================================

for col in [
    "xmin",
    "ymin",
    "xmax",
    "ymax"
]:

    yolo_df[col] = (
        yolo_df[col]
        .clip(0.0, 1.0)
    )

# ============================================================================
# OBJECT → COMMON VOCABULARY
# ============================================================================

yolo_df["resolved_object_id"] = (
    yolo_df["object_name"]
    .map(normalized_object_to_id)
)

unknown_mask = (
    yolo_df["resolved_object_id"]
    .isna()
)

if unknown_mask.any():

    unknown_names = sorted(
        yolo_df.loc[
            unknown_mask,
            "object_name"
        ].unique()
    )

    raise ValueError(
        "Unknown YOLO26 classes:\n"
        + "\n".join(
            unknown_names[:50]
        )
    )

yolo_df["resolved_object_id"] = (
    yolo_df["resolved_object_id"]
    .astype(np.int64)
)

# ============================================================================
# COMPLETE 39,874-IMAGE ORDER
# ============================================================================

structured_image_ids = np.asarray(
    sorted(
        set(
            caption_image_ids.tolist()
        )
    ),
    dtype=str
)

assert len(
    structured_image_ids
) == 39874

image_to_row = {
    image_id: i
    for i, image_id
    in enumerate(
        structured_image_ids
    )
}

# ============================================================================
# STRUCTURED ARRAYS
# ============================================================================

yolo26_object_ids = np.zeros(
    (39874, TOP_K),
    dtype=np.int64
)

yolo26_spatial = np.zeros(
    (39874, TOP_K, SPATIAL_DIM),
    dtype=np.float32
)

yolo26_confidence = np.zeros(
    (39874, TOP_K),
    dtype=np.float32
)

yolo26_object_mask = np.zeros(
    (39874, TOP_K),
    dtype=bool
)

# ============================================================================
# TOP-K DETECTIONS
# ============================================================================

yolo_df = yolo_df.sort_values(
    [
        "image_id",
        "confidence"
    ],
    ascending=[
        True,
        False
    ]
)

for image_id, group in yolo_df.groupby(
    "image_id",
    sort=False
):

    row = image_to_row.get(
        image_id
    )

    if row is None:
        continue

    group = group.head(
        TOP_K
    )

    for j, r in enumerate(
        group.itertuples(
            index=False
        )
    ):

        xmin = float(r.xmin)
        ymin = float(r.ymin)
        xmax = float(r.xmax)
        ymax = float(r.ymax)

        left = min(
            xmin,
            xmax
        )

        right = max(
            xmin,
            xmax
        )

        top = min(
            ymin,
            ymax
        )

        bottom = max(
            ymin,
            ymax
        )

        center_x = (
            left + right
        ) / 2.0

        center_y = (
            top + bottom
        ) / 2.0

        box_width = max(
            0.0,
            right - left
        )

        box_height = max(
            0.0,
            bottom - top
        )

        box_area = (
            box_width
            * box_height
        )

        yolo26_object_ids[
            row,
            j
        ] = int(
            r.resolved_object_id
        )

        yolo26_spatial[
            row,
            j
        ] = [
            left,
            top,
            right,
            bottom,
            center_x,
            center_y,
            box_width,
            box_height,
            box_area
        ]

        yolo26_confidence[
            row,
            j
        ] = float(
            r.confidence
        )

        yolo26_object_mask[
            row,
            j
        ] = True

# ============================================================================
# SAVE PERMANENTLY
# ============================================================================

np.savez_compressed(
    YOLO26_FEATURE_FILE,
    image_ids=structured_image_ids,
    object_ids=yolo26_object_ids,
    boxes=yolo26_spatial,
    confidences=yolo26_confidence,
    object_mask=yolo26_object_mask
)

# ============================================================================
# VERIFY
# ============================================================================

assert yolo26_object_ids.shape == (
    39874,
    10
)

assert yolo26_spatial.shape == (
    39874,
    10,
    9
)

assert yolo26_confidence.shape == (
    39874,
    10
)

assert yolo26_object_mask.shape == (
    39874,
    10
)

assert np.isfinite(
    yolo26_spatial
).all()

assert np.isfinite(
    yolo26_confidence
).all()

assert yolo26_spatial.min() >= 0
assert yolo26_spatial.max() <= 1

images_with_objects = int(
    yolo26_object_mask
    .any(axis=1)
    .sum()
)

zero_object_images = (
    39874
    - images_with_objects
)

print()
print("=" * 75)
print("🔥 YOLO26 STRUCTURED FEATURES COMPLETE")
print("=" * 75)

print(
    f"Detection rows used:    {len(yolo_df):,}"
)

print(
    f"Dataset images:         {39874:,}"
)

print(
    f"Images with detections: "
    f"{images_with_objects:,}"
)

print(
    f"Zero-object images:     {zero_object_images:,}"
)

print(
    f"Object IDs:             "
    f"{yolo26_object_ids.shape}"
)

print(
    f"Spatial:                "
    f"{yolo26_spatial.shape}"
)

print(
    f"Confidence:             "
    f"{yolo26_confidence.shape}"
)

print()
print(
    f"Saved permanently:\n"
    f"{YOLO26_FEATURE_FILE}"
)

print()
print("✓ Existing YOLO26 CSV used")
print("✓ No YOLO26 inference")
print("✓ No local dataset")
print("✓ No cache")
print("✓ Duplicate Drive paths resolved")
print("✓ Each unique image read once")
print("✓ TOP-K = 10")
print("✓ Confidence threshold = 0.30")
print("=" * 75)
print("CELL 3 COMPLETE")
print("=" * 75)

CELL 3 — FAST YOLO26s STRUCTURED OBJECT FEATURES
Reading YOLO26 detection CSV...
Detection rows loaded: 239,048
Rows after confidence >= 0.3: 216,324
Unique detection images required: 39,737

Searching permanent Google Drive images...
Drive image search complete in 36.9s
Unique image filenames found: 39,737
Duplicate filename groups: 31,693
✓ Duplicate paths resolved using shallowest Drive path

Reading dimensions for 39,737 unique images...
  5,000/39,737 | 27 img/s
  10,000/39,737 | 33 img/s
  15,000/39,737 | 26 img/s
  20,000/39,737 | 24 img/s
  25,000/39,737 | 23 img/s
  30,000/39,737 | 22 img/s
  35,000/39,737 | 22 img/s
  39,737/39,737 | 21 img/s

Maximum raw box coordinate: 500.0000
Absolute pixel coordinates detected.

🔥 YOLO26 STRUCTURED FEATURES COMPLETE
Detection rows used:    216,324
Dataset images:         39,874
Images with detections: 39,737
Zero-object images:     137
Object IDs:             (39874, 10)
Spatial:                (39874, 10, 9)
Confidence:             (398

In [ ]:
# ===========================================================================
# CELL 4 — YOLO26 DATASET + OBJECT-AWARE TRANSFORMER
# ===========================================================================

print("=" * 75)
print("CELL 4 — YOLO26 DATASET + OBJECT-AWARE TRANSFORMER")
print("=" * 75)

# ---------------------------------------------------------------------------
# Load YOLO26 structured features
# ---------------------------------------------------------------------------

yolo26_features = np.load(
    YOLO26_FEATURE_FILE,
    allow_pickle=True
)

yolo26_image_ids = yolo26_features["image_ids"].astype(str)
yolo26_object_ids = yolo26_features["object_ids"].astype(np.int64)
yolo26_spatial = yolo26_features["boxes"].astype(np.float32)
yolo26_confidence = yolo26_features["confidences"].astype(np.float32)
yolo26_object_mask = yolo26_features["object_mask"].astype(bool)

print("YOLO26 structured features:")
print("  Images:     ", len(yolo26_image_ids))
print("  Object IDs: ", yolo26_object_ids.shape)
print("  Spatial:    ", yolo26_spatial.shape)
print("  Confidence: ", yolo26_confidence.shape)
print("  Mask:       ", yolo26_object_mask.shape)

# ---------------------------------------------------------------------------
# Load caption sequences DIRECTLY from permanent Drive file
# ---------------------------------------------------------------------------

caption_data = np.load(
    CAPTION_SEQ_FILE,
    allow_pickle=True
)

caption_image_ids = caption_data["image_ids"].astype(str)
caption_sequences = caption_data["caption_ids"].astype(np.int64)

print("\nCaption data:")
print("  Images/caption records:", len(caption_image_ids))
print("  Caption sequences:     ", caption_sequences.shape)

assert len(caption_image_ids) == len(caption_sequences)
assert len(caption_image_ids) == 199_369

# ---------------------------------------------------------------------------
# Image → YOLO26 feature index
# ---------------------------------------------------------------------------

yolo26_image_to_idx = {
    image_id: idx
    for idx, image_id in enumerate(yolo26_image_ids)
}

# ---------------------------------------------------------------------------
# Caption record → YOLO26 feature index
# ---------------------------------------------------------------------------

missing_images = [
    image_id
    for image_id in caption_image_ids
    if image_id not in yolo26_image_to_idx
]

assert len(missing_images) == 0, (
    f"Missing YOLO26 features for {len(missing_images)} caption images. "
    f"Examples: {missing_images[:5]}"
)

caption_to_yolo26_idx = np.array(
    [
        yolo26_image_to_idx[image_id]
        for image_id in caption_image_ids
    ],
    dtype=np.int64
)

print("Caption records mapped:", len(caption_to_yolo26_idx))
print("✓ All caption images have YOLO26 structured features")

# ===========================================================================
# DATASET
# ===========================================================================

class YOLO26ObjectCaptionDataset(Dataset):

    def __init__(self, caption_indices):
        self.caption_indices = np.asarray(
            caption_indices,
            dtype=np.int64
        )

    def __len__(self):
        return len(self.caption_indices)

    def __getitem__(self, idx):

        caption_idx = self.caption_indices[idx]
        feature_idx = caption_to_yolo26_idx[caption_idx]

        return (
            torch.from_numpy(
                yolo26_object_ids[feature_idx]
            ).long(),

            torch.from_numpy(
                yolo26_spatial[feature_idx]
            ).float(),

            torch.from_numpy(
                yolo26_confidence[feature_idx]
            ).float(),

            torch.from_numpy(
                yolo26_object_mask[feature_idx]
            ).bool(),

            torch.from_numpy(
                caption_sequences[caption_idx]
            ).long()
        )


train_dataset = YOLO26ObjectCaptionDataset(
    train_caption_indices
)

val_dataset = YOLO26ObjectCaptionDataset(
    val_caption_indices
)

test_dataset = YOLO26ObjectCaptionDataset(
    test_caption_indices
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("\nDataset sizes:")
print("  Train:", len(train_dataset))
print("  Val:  ", len(val_dataset))
print("  Test: ", len(test_dataset))

assert len(train_dataset) == 159_494
assert len(val_dataset) == 19_935
assert len(test_dataset) == 19_940

# ===========================================================================
# OBJECT ENCODER
# ===========================================================================

class ObjectEncoder(nn.Module):

    def __init__(
        self,
        object_vocab_size,
        d_model=256
    ):
        super().__init__()

        self.object_embedding = nn.Embedding(
            object_vocab_size,
            d_model,
            padding_idx=0
        )

        self.spatial_mlp = nn.Sequential(
            nn.Linear(SPATIAL_DIM, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.confidence_mlp = nn.Sequential(
            nn.Linear(1, 128),
            nn.ReLU(),
            nn.Linear(128, d_model)
        )

        self.fusion = nn.Sequential(
            nn.Linear(d_model * 3, d_model),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.LayerNorm(d_model)
        )

    def forward(
        self,
        object_ids,
        spatial,
        confidence
    ):

        object_embedding = self.object_embedding(
            object_ids
        )

        spatial_embedding = self.spatial_mlp(
            spatial
        )

        confidence_embedding = self.confidence_mlp(
            confidence.unsqueeze(-1)
        )

        combined = torch.cat(
            [
                object_embedding,
                spatial_embedding,
                confidence_embedding
            ],
            dim=-1
        )

        return self.fusion(combined)


# ===========================================================================
# POSITIONAL ENCODING
# ===========================================================================

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len
    ):
        super().__init__()

        position = torch.arange(
            max_len
        ).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ).float()
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


# ===========================================================================
# CAPTION EMBEDDING
# ===========================================================================

class CaptionEmbedding(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_ID
        )

        self.position = PositionalEncoding(
            d_model,
            MAX_CAPTION_LENGTH
        )

    def forward(self, x):
        return self.position(
            self.embedding(x)
        )


# ===========================================================================
# OBJECT-AWARE TRANSFORMER
# ===========================================================================

class ObjectAwareTransformer(nn.Module):

    def __init__(
        self,
        object_vocab_size,
        caption_vocab_size,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=1024,
        dropout=0.2,
        max_caption_length=32
    ):
        super().__init__()

        self.object_encoder = ObjectEncoder(
            object_vocab_size,
            d_model
        )

        self.object_position = PositionalEncoding(
            d_model,
            TOP_K
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )

        self.caption_embedding = CaptionEmbedding(
            caption_vocab_size,
            d_model
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_decoder_layers
        )

        self.output_projection = nn.Linear(
            d_model,
            caption_vocab_size
        )

    def generate_causal_mask(
        self,
        size,
        device
    ):
        return torch.triu(
            torch.ones(
                size,
                size,
                device=device,
                dtype=torch.bool
            ),
            diagonal=1
        )

    def forward(
        self,
        object_ids,
        spatial,
        confidence,
        object_mask,
        decoder_input
    ):

        object_tokens = self.object_encoder(
            object_ids,
            spatial,
            confidence
        )

        object_tokens = self.object_position(
            object_tokens
        )

        memory = self.transformer_encoder(
            object_tokens,
            src_key_padding_mask=~object_mask
        )

        decoder_tokens = self.caption_embedding(
            decoder_input
        )

        target_mask = self.generate_causal_mask(
            decoder_input.size(1),
            decoder_input.device
        )

        decoded = self.transformer_decoder(
            decoder_tokens,
            memory,
            tgt_mask=target_mask,
            memory_key_padding_mask=~object_mask
        )

        logits = self.output_projection(
            decoded
        )

        return logits


# ===========================================================================
# CREATE MODEL
# ===========================================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ObjectAwareTransformer(
    object_vocab_size=len(object_to_id),
    caption_vocab_size=len(caption_to_id),
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_caption_length=MAX_CAPTION_LENGTH
).to(device)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

print("\nModel:")
print("  Device:     ", device)
print("  Parameters: ", f"{parameter_count:,}")

assert parameter_count == 12_347_258, (
    f"Unexpected parameter count: {parameter_count:,}"
)

print("✓ Parameter count matches RT-DETR Transformer")
print("✓ YOLO26 structured features connected")
print("✓ Object + spatial + confidence embeddings")
print("✓ Transformer encoder + decoder")
print("✓ Architecture matches RT-DETR downstream model")

print("=" * 75)
print("CELL 4 COMPLETE")
print("=" * 75)

CELL 4 — YOLO26 DATASET + OBJECT-AWARE TRANSFORMER
YOLO26 structured features:
  Images:      39874
  Object IDs:  (39874, 10)
  Spatial:     (39874, 10, 9)
  Confidence:  (39874, 10)
  Mask:        (39874, 10)

Caption data:
  Images/caption records: 199369
  Caption sequences:      (199369, 32)
Caption records mapped: 199369
✓ All caption images have YOLO26 structured features

Dataset sizes:
  Train: 159494
  Val:   19935
  Test:  19940


NameError: name 'MAX_CAPTION_LENGTH' is not defined

In [ ]:
# ===========================================================================
# CELL 5 — YOLO26 MODEL SANITY CHECK + LOSS SETUP
# ===========================================================================

print("=" * 75)
print("CELL 5 — YOLO26 MODEL SANITY CHECK + LOSS SETUP")
print("=" * 75)

# ---------------------------------------------------------------------------
# Safety: make sure the model exists
# ---------------------------------------------------------------------------

if "model" not in globals():

    print("Model not found in memory.")
    print("Creating YOLO26 Object-Aware Transformer now...")

    MAX_CAPTION_LENGTH = 32

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model = ObjectAwareTransformer(
        object_vocab_size=len(object_to_id),
        caption_vocab_size=len(caption_to_id),
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        num_decoder_layers=NUM_DECODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        max_caption_length=MAX_CAPTION_LENGTH
    ).to(device)

    parameter_count = sum(
        p.numel()
        for p in model.parameters()
    )

    assert parameter_count == 12_347_258, (
        f"Unexpected parameter count: {parameter_count:,}"
    )

    print(f"✓ Model created: {parameter_count:,} parameters")

else:

    print("✓ Model already exists in memory")

# ---------------------------------------------------------------------------
# Get one training batch
# ---------------------------------------------------------------------------

batch = next(iter(train_loader))

object_ids_batch = batch[0].to(
    device,
    non_blocking=True
)

spatial_batch = batch[1].to(
    device,
    non_blocking=True
)

confidence_batch = batch[2].to(
    device,
    non_blocking=True
)

object_mask_batch = batch[3].to(
    device,
    non_blocking=True
)

caption_batch = batch[4].to(
    device,
    non_blocking=True
)

print("\nBatch shapes:")
print("  Object IDs:  ", object_ids_batch.shape)
print("  Spatial:     ", spatial_batch.shape)
print("  Confidence:  ", confidence_batch.shape)
print("  Object mask: ", object_mask_batch.shape)
print("  Captions:    ", caption_batch.shape)

# ---------------------------------------------------------------------------
# Teacher forcing
# ---------------------------------------------------------------------------

decoder_input = caption_batch[:, :-1]
decoder_target = caption_batch[:, 1:]

print("\nDecoder:")
print("  Input:  ", decoder_input.shape)
print("  Target: ", decoder_target.shape)

# ---------------------------------------------------------------------------
# Forward pass
# ---------------------------------------------------------------------------

model.eval()

with torch.no_grad():

    logits = model(
        object_ids_batch,
        spatial_batch,
        confidence_batch,
        object_mask_batch,
        decoder_input
    )

print("\nForward pass:")
print("  Logits:", logits.shape)

expected_shape = (
    caption_batch.size(0),
    MAX_CAPTION_LENGTH - 1,
    len(caption_to_id)
)

assert logits.shape == expected_shape, (
    f"Unexpected logits shape: {logits.shape}; "
    f"expected {expected_shape}"
)

assert torch.isfinite(logits).all(), (
    "NaN/Inf detected in model output"
)

print("✓ Forward pass successful")
print("✓ Logits shape correct")
print("✓ No NaN/Inf detected")

# ---------------------------------------------------------------------------
# Loss
# ---------------------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    label_smoothing=LABEL_SMOOTHING
)

loss = criterion(
    logits.reshape(-1, logits.size(-1)),
    decoder_target.reshape(-1)
)

assert torch.isfinite(loss), (
    "Initial loss is NaN/Inf"
)

print("\nInitial loss:", f"{loss.item():.6f}")
print("✓ CrossEntropyLoss initialized")
print(f"✓ Padding ID ignored: {PAD_ID}")
print(f"✓ Label smoothing: {LABEL_SMOOTHING}")

# ---------------------------------------------------------------------------
# GPU memory
# ---------------------------------------------------------------------------

if torch.cuda.is_available():

    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)

    print("\nGPU memory:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")

print("=" * 75)
print("CELL 5 COMPLETE")
print("=" * 75)

CELL 5 — YOLO26 MODEL SANITY CHECK + LOSS SETUP
Model not found in memory.
Creating YOLO26 Object-Aware Transformer now...
✓ Model created: 12,347,258 parameters

Batch shapes:
  Object IDs:   torch.Size([64, 10])
  Spatial:      torch.Size([64, 10, 9])
  Confidence:   torch.Size([64, 10])
  Object mask:  torch.Size([64, 10])
  Captions:     torch.Size([64, 32])

Decoder:
  Input:   torch.Size([64, 31])
  Target:  torch.Size([64, 31])

Forward pass:
  Logits: torch.Size([64, 31, 12666])
✓ Forward pass successful
✓ Logits shape correct
✓ No NaN/Inf detected

Initial loss: 10.371355
✓ CrossEntropyLoss initialized
✓ Padding ID ignored: 0
✓ Label smoothing: 0.1

GPU memory:
  Allocated: 0.15 GB
  Reserved:  0.35 GB
CELL 5 COMPLETE


In [ ]:
# ===========================================================================
# CELL 6 FIX — YOLO26 TRAINING CONFIGURATION
# ===========================================================================

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 4

print("=" * 75)
print("YOLO26 TRAINING CONFIGURATION")
print("=" * 75)

print(f"Learning rate:             {LEARNING_RATE}")
print(f"Weight decay:              {WEIGHT_DECAY}")
print(f"Label smoothing:           {LABEL_SMOOTHING}")
print(f"Gradient clipping:         {MAX_GRAD_NORM}")
print(f"Early stopping patience:   {EARLY_STOPPING_PATIENCE}")

print("✓ Training configuration loaded")

YOLO26 TRAINING CONFIGURATION
Learning rate:             0.0001
Weight decay:              0.0001
Label smoothing:           0.1
Gradient clipping:         1.0
Early stopping patience:   4
✓ Training configuration loaded


In [ ]:
# ===========================================================================
# CELL 6 — YOLO26 OPTIMIZER + SCHEDULER + TRAINING FUNCTIONS
# ===========================================================================

print("=" * 75)
print("CELL 6 — YOLO26 OPTIMIZER + SCHEDULER + TRAINING FUNCTIONS")
print("=" * 75)

# ---------------------------------------------------------------------------
# Required safety checks
# ---------------------------------------------------------------------------

assert "model" in globals(), "Model is missing. Run Cell 5 first."
assert "train_loader" in globals(), "Train loader is missing."
assert "val_loader" in globals(), "Validation loader is missing."
assert "criterion" in globals(), "Loss function is missing."

# ---------------------------------------------------------------------------
# Optimizer
# ---------------------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ---------------------------------------------------------------------------
# Scheduler
# ---------------------------------------------------------------------------

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)

print("Optimizer: AdamW")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Weight decay:  {WEIGHT_DECAY}")

print("\nScheduler: ReduceLROnPlateau")
print("Factor:   0.5")
print("Patience: 1")

# ===========================================================================
# TRAIN ONE EPOCH
# ===========================================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0.0
    total_batches = 0

    start_time = time.time()

    for batch_idx, batch in enumerate(loader):

        object_ids = batch[0].to(
            device,
            non_blocking=True
        )

        spatial = batch[1].to(
            device,
            non_blocking=True
        )

        confidence = batch[2].to(
            device,
            non_blocking=True
        )

        object_mask = batch[3].to(
            device,
            non_blocking=True
        )

        captions = batch[4].to(
            device,
            non_blocking=True
        )

        decoder_input = captions[:, :-1]
        decoder_target = captions[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            object_ids,
            spatial,
            confidence,
            object_mask,
            decoder_input
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            decoder_target.reshape(-1)
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss at batch "
                f"{batch_idx + 1}: {loss.item()}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )

        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

        if (batch_idx + 1) % 100 == 0:

            elapsed = time.time() - start_time

            print(
                f"  Batch {batch_idx + 1:>4}/{len(loader)}"
                f" | Loss {loss.item():.4f}"
                f" | Time {elapsed / 60:.1f} min"
            )

    return total_loss / total_batches


# ===========================================================================
# VALIDATION
# ===========================================================================

def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():

        for batch in loader:

            object_ids = batch[0].to(
                device,
                non_blocking=True
            )

            spatial = batch[1].to(
                device,
                non_blocking=True
            )

            confidence = batch[2].to(
                device,
                non_blocking=True
            )

            object_mask = batch[3].to(
                device,
                non_blocking=True
            )

            captions = batch[4].to(
                device,
                non_blocking=True
            )

            decoder_input = captions[:, :-1]
            decoder_target = captions[:, 1:]

            logits = model(
                object_ids,
                spatial,
                confidence,
                object_mask,
                decoder_input
            )

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                decoder_target.reshape(-1)
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite validation loss: {loss.item()}"
                )

            total_loss += loss.item()
            total_batches += 1

    return total_loss / total_batches


# ===========================================================================
# TRAINING STATE
# ===========================================================================

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "learning_rate": [],
    "epoch_time_seconds": []
}

best_val_loss = float("inf")
epochs_without_improvement = 0

print("\n✓ train_one_epoch() ready")
print("✓ validate_one_epoch() ready")
print(f"✓ Gradient clipping: {MAX_GRAD_NORM}")
print(f"✓ Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print("✓ Training history initialized")

print("=" * 75)
print("CELL 6 COMPLETE")
print("=" * 75)

CELL 6 — YOLO26 OPTIMIZER + SCHEDULER + TRAINING FUNCTIONS
Optimizer: AdamW
Learning rate: 0.0001
Weight decay:  0.0001

Scheduler: ReduceLROnPlateau
Factor:   0.5
Patience: 1

✓ train_one_epoch() ready
✓ validate_one_epoch() ready
✓ Gradient clipping: 1.0
✓ Early stopping patience: 4
✓ Training history initialized
CELL 6 COMPLETE


In [ ]:
# ===========================================================================
# CELL 7 FIX — COMPLETE TRAINING CONFIGURATION
# ===========================================================================

EPOCHS = 20
BATCH_SIZE = 64

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 4

print("=" * 75)
print("YOLO26 TRAINING CONFIGURATION")
print("=" * 75)

print(f"Epochs:                    {EPOCHS}")
print(f"Batch size:                {BATCH_SIZE}")
print(f"Learning rate:             {LEARNING_RATE}")
print(f"Weight decay:              {WEIGHT_DECAY}")
print(f"Label smoothing:           {LABEL_SMOOTHING}")
print(f"Gradient clipping:         {MAX_GRAD_NORM}")
print(f"Early stopping patience:   {EARLY_STOPPING_PATIENCE}")

print("✓ Complete training configuration loaded")

YOLO26 TRAINING CONFIGURATION
Epochs:                    20
Batch size:                64
Learning rate:             0.0001
Weight decay:              0.0001
Label smoothing:           0.1
Gradient clipping:         1.0
Early stopping patience:   4
✓ Complete training configuration loaded


In [ ]:
# ===========================================================================
# CELL 7 FIX — HANDLE ZERO-OBJECT IMAGES WITHOUT NaN
# ===========================================================================

print("=" * 75)
print("CELL 7 FIX — ZERO-OBJECT ATTENTION SAFETY")
print("=" * 75)

# ---------------------------------------------------------------------------
# Replace the model forward method
# ---------------------------------------------------------------------------

def object_aware_forward(
    self,
    object_ids,
    spatial,
    confidence,
    object_mask,
    decoder_input
):

    # ---------------------------------------------------------------
    # Encode objects
    # ---------------------------------------------------------------

    object_tokens = self.object_encoder(
        object_ids,
        spatial,
        confidence
    )

    object_tokens = self.object_position(
        object_tokens
    )

    # ---------------------------------------------------------------
    # IMPORTANT:
    # Transformer cannot handle a sample where every source token
    # is masked. Give zero-object images one valid PAD token.
    # ---------------------------------------------------------------

    safe_object_mask = object_mask.clone()

    empty_samples = ~safe_object_mask.any(dim=1)

    if empty_samples.any():

        safe_object_mask[empty_samples, 0] = True

        # Keep the artificial token numerically harmless.
        object_tokens = object_tokens.clone()

        object_tokens[empty_samples, 0] = 0.0

    # ---------------------------------------------------------------
    # Encoder
    # ---------------------------------------------------------------

    memory = self.transformer_encoder(
        object_tokens,
        src_key_padding_mask=~safe_object_mask
    )

    # ---------------------------------------------------------------
    # Decoder
    # ---------------------------------------------------------------

    decoder_tokens = self.caption_embedding(
        decoder_input
    )

    target_mask = self.generate_causal_mask(
        decoder_input.size(1),
        decoder_input.device
    )

    decoded = self.transformer_decoder(
        decoder_tokens,
        memory,
        tgt_mask=target_mask,
        memory_key_padding_mask=~safe_object_mask
    )

    logits = self.output_projection(
        decoded
    )

    return logits


# Bind fixed forward method
ObjectAwareTransformer.forward = object_aware_forward

print("✓ Zero-object attention handling installed")
print("✓ Empty object sequences receive one safe zero token")
print("✓ Real object masks remain unchanged")
print("✓ No dataset regeneration required")

print("=" * 75)
print("CELL 7 FIX COMPLETE")
print("=" * 75)

CELL 7 FIX — ZERO-OBJECT ATTENTION SAFETY
✓ Zero-object attention handling installed
✓ Empty object sequences receive one safe zero token
✓ Real object masks remain unchanged
✓ No dataset regeneration required
CELL 7 FIX COMPLETE


In [ ]:
# ===========================================================================
# CELL 7 FIX TEST — VALIDATION NUMERICAL CHECK
# ===========================================================================

print("=" * 75)
print("CELL 7 FIX TEST — VALIDATION")
print("=" * 75)

model.eval()

validation_loss = validate_one_epoch(
    model,
    val_loader,
    criterion,
    device
)

print("\nValidation loss:", f"{validation_loss:.6f}")

assert math.isfinite(validation_loss), (
    f"Validation loss is still non-finite: {validation_loss}"
)

print("✓ Validation loss is finite")
print("✓ Zero-object handling works")
print("=" * 75)
print("CELL 7 FIX TEST COMPLETE")
print("=" * 75)

CELL 7 FIX TEST — VALIDATION

Validation loss: 4.212931
✓ Validation loss is finite
✓ Zero-object handling works
CELL 7 FIX TEST COMPLETE


In [ ]:
# ===========================================================================
# CELL 7 FIX — YOLO26 CONFIDENCE THRESHOLD
# ===========================================================================

CONFIDENCE_THRESHOLD = 0.30

print("CONFIDENCE_THRESHOLD =", CONFIDENCE_THRESHOLD)
print("✓ Training configuration complete")

CONFIDENCE_THRESHOLD = 0.3
✓ Training configuration complete


In [ ]:
# ===========================================================================
# CELL 7 — YOLO26 FINAL TRAINING — CLEAN OUTPUT
# ===========================================================================

print("=" * 75)
print("CELL 7 — YOLO26 OBJECT-AWARE TRANSFORMER TRAINING")
print("=" * 75)

assert "model" in globals()
assert "optimizer" in globals()
assert "scheduler" in globals()
assert "criterion" in globals()

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Clean history for fresh training
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "learning_rate": [],
    "epoch_time_seconds": []
}

best_val_loss = float("inf")
epochs_without_improvement = 0

training_start = time.time()

# ===========================================================================
# CLEAN TRAINING FUNCTIONS
# ===========================================================================

def train_one_epoch_clean(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0.0
    total_batches = 0

    for batch in loader:

        object_ids = batch[0].to(device, non_blocking=True)
        spatial = batch[1].to(device, non_blocking=True)
        confidence = batch[2].to(device, non_blocking=True)
        object_mask = batch[3].to(device, non_blocking=True)
        captions = batch[4].to(device, non_blocking=True)

        decoder_input = captions[:, :-1]
        decoder_target = captions[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            object_ids,
            spatial,
            confidence,
            object_mask,
            decoder_input
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            decoder_target.reshape(-1)
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss: {loss.item()}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )

        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / total_batches


def validate_one_epoch_clean(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():

        for batch in loader:

            object_ids = batch[0].to(device, non_blocking=True)
            spatial = batch[1].to(device, non_blocking=True)
            confidence = batch[2].to(device, non_blocking=True)
            object_mask = batch[3].to(device, non_blocking=True)
            captions = batch[4].to(device, non_blocking=True)

            decoder_input = captions[:, :-1]
            decoder_target = captions[:, 1:]

            logits = model(
                object_ids,
                spatial,
                confidence,
                object_mask,
                decoder_input
            )

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                decoder_target.reshape(-1)
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite validation loss: {loss.item()}"
                )

            total_loss += loss.item()
            total_batches += 1

    return total_loss / total_batches


# ===========================================================================
# TRAIN
# ===========================================================================

for epoch in range(1, EPOCHS + 1):

    epoch_start = time.time()

    train_loss = train_one_epoch_clean(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss = validate_one_epoch_clean(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    epoch_time = time.time() - epoch_start

    history["epoch"].append(epoch)
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_loss))
    history["learning_rate"].append(float(current_lr))
    history["epoch_time_seconds"].append(float(epoch_time))

    improved = val_loss < best_val_loss

    if improved:
        best_val_loss = val_loss
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    # -----------------------------------------------------------------------
    # CHECKPOINT
    # -----------------------------------------------------------------------

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_loss": best_val_loss,
        "history": history,
        "config": {
            "model": "YOLO26s Object-Aware Transformer",
            "d_model": D_MODEL,
            "nhead": NHEAD,
            "encoder_layers": NUM_ENCODER_LAYERS,
            "decoder_layers": NUM_DECODER_LAYERS,
            "dim_feedforward": DIM_FEEDFORWARD,
            "dropout": DROPOUT,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "label_smoothing": LABEL_SMOOTHING,
            "max_grad_norm": MAX_GRAD_NORM,
            "max_caption_length": MAX_CAPTION_LENGTH,
            "top_k": TOP_K,
            "confidence_threshold": CONFIDENCE_THRESHOLD,
            "caption_vocab_size": len(caption_to_id),
            "object_vocab_size": len(object_to_id)
        }
    }

    torch.save(
        checkpoint,
        LAST_CHECKPOINT
    )

    if improved:

        torch.save(
            checkpoint,
            BEST_CHECKPOINT
        )

        marker = " ★ BEST"

    else:

        marker = ""

    # -----------------------------------------------------------------------
    # SAVE HISTORY
    # -----------------------------------------------------------------------

    history_path = (
        OUTPUT_DIR /
        "yolo26_training_history.json"
    )

    with open(
        history_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )

    # -----------------------------------------------------------------------
    # CLEAN OUTPUT — NO BATCH LOGS
    # -----------------------------------------------------------------------

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train: {train_loss:.6f} | "
        f"Val: {val_loss:.6f} | "
        f"LR: {current_lr:.2e} | "
        f"Time: {epoch_time / 60:.2f} min"
        f"{marker}"
    )

    # -----------------------------------------------------------------------
    # EARLY STOPPING
    # -----------------------------------------------------------------------

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:

        print(
            f"\nEarly stopping after {epoch} epochs "
            f"(patience={EARLY_STOPPING_PATIENCE})."
        )

        break


# ===========================================================================
# COMPLETE
# ===========================================================================

total_time = time.time() - training_start

print("\n" + "=" * 75)
print("🔥 YOLO26 TRAINING COMPLETE")
print("=" * 75)

print(f"Epochs completed:     {len(history['epoch'])}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Training time:        {total_time / 3600:.2f} hours")

print("\nDrive files:")
print(f"BEST CHECKPOINT: {BEST_CHECKPOINT}")
print(f"LAST CHECKPOINT: {LAST_CHECKPOINT}")
print(f"HISTORY:         {history_path}")

print("\n✓ Training completed")
print("✓ Best model saved")
print("✓ Last model saved")
print("✓ History saved")
print("✓ No batch output")
print("=" * 75)
print("CELL 7 COMPLETE")
print("=" * 75)

CELL 7 — YOLO26 OBJECT-AWARE TRANSFORMER TRAINING
Epoch 01/20 | Train: 4.044116 | Val: 3.930323 | LR: 1.00e-04 | Time: 2.02 min ★ BEST
Epoch 02/20 | Train: 3.949843 | Val: 3.874627 | LR: 1.00e-04 | Time: 2.01 min ★ BEST
Epoch 03/20 | Train: 3.881364 | Val: 3.831334 | LR: 1.00e-04 | Time: 2.03 min ★ BEST
Epoch 04/20 | Train: 3.827264 | Val: 3.796422 | LR: 1.00e-04 | Time: 2.02 min ★ BEST
Epoch 05/20 | Train: 3.781700 | Val: 3.770936 | LR: 1.00e-04 | Time: 2.03 min ★ BEST
Epoch 06/20 | Train: 3.742883 | Val: 3.747593 | LR: 1.00e-04 | Time: 2.03 min ★ BEST
Epoch 07/20 | Train: 3.709492 | Val: 3.729546 | LR: 1.00e-04 | Time: 2.02 min ★ BEST
Epoch 08/20 | Train: 3.679080 | Val: 3.717542 | LR: 1.00e-04 | Time: 2.04 min ★ BEST
Epoch 09/20 | Train: 3.652050 | Val: 3.703054 | LR: 1.00e-04 | Time: 2.02 min ★ BEST
Epoch 10/20 | Train: 3.626764 | Val: 3.690763 | LR: 1.00e-04 | Time: 2.02 min ★ BEST
Epoch 11/20 | Train: 3.604020 | Val: 3.681834 | LR: 1.00e-04 | Time: 2.03 min ★ BEST
Epoch 12/20 | T

In [ ]:
# ===========================================================================
# CELL 8 — YOLO26 BEST MODEL LOADING + TEST CAPTION GENERATION
# ===========================================================================

print("=" * 75)
print("CELL 8 — YOLO26 BEST MODEL + TEST CAPTION GENERATION")
print("=" * 75)

# ---------------------------------------------------------------------------
# Load BEST checkpoint
# ---------------------------------------------------------------------------

assert BEST_CHECKPOINT.exists(), (
    f"Best checkpoint not found:\n{BEST_CHECKPOINT}"
)

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

print("✓ Best checkpoint loaded")
print(f"  Epoch: {checkpoint['epoch']}")
print(f"  Best validation loss: {checkpoint['best_val_loss']:.6f}")

# ---------------------------------------------------------------------------
# Special token IDs
# ---------------------------------------------------------------------------

PAD_ID = int(caption_to_id["<pad>"])
START_ID = int(caption_to_id["<start>"])
END_ID = int(caption_to_id["<end>"])
UNK_ID = int(caption_to_id["<unk>"])

print("\nSpecial tokens:")
print(f"  PAD:   {PAD_ID}")
print(f"  START: {START_ID}")
print(f"  END:   {END_ID}")
print(f"  UNK:   {UNK_ID}")

# ---------------------------------------------------------------------------
# ID → word
# ---------------------------------------------------------------------------

id_to_caption = {
    int(v): str(k)
    for k, v in caption_to_id.items()
}

# ---------------------------------------------------------------------------
# Greedy caption generation
# ---------------------------------------------------------------------------

def generate_caption_greedy(
    model,
    object_ids,
    spatial,
    confidence,
    object_mask,
    max_length=MAX_CAPTION_LENGTH
):

    model.eval()

    # Add batch dimension
    object_ids = object_ids.unsqueeze(0).to(device)
    spatial = spatial.unsqueeze(0).to(device)
    confidence = confidence.unsqueeze(0).to(device)
    object_mask = object_mask.unsqueeze(0).to(device)

    generated = torch.tensor(
        [[START_ID]],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        for _ in range(max_length - 1):

            logits = model(
                object_ids,
                spatial,
                confidence,
                object_mask,
                generated
            )

            next_token = torch.argmax(
                logits[:, -1, :],
                dim=-1,
                keepdim=True
            )

            generated = torch.cat(
                [
                    generated,
                    next_token
                ],
                dim=1
            )

            if next_token.item() == END_ID:
                break

    token_ids = generated[0].tolist()

    words = []

    for token_id in token_ids:

        if token_id in (
            PAD_ID,
            START_ID,
            END_ID
        ):
            continue

        word = id_to_caption.get(
            token_id,
            "<unk>"
        )

        words.append(word)

    return " ".join(words)


# ===========================================================================
# TEST GENERATION
# ===========================================================================

test_image_ids = np.array(
    sorted(test_image_ids_set)
    if "test_image_ids_set" in globals()
    else sorted(
        test_image_ids
    ),
    dtype=str
)

print("\nTest images:", len(test_image_ids))

assert len(test_image_ids) == 3988, (
    f"Expected 3988 test images, got {len(test_image_ids)}"
)

# ---------------------------------------------------------------------------
# Feature lookup
# ---------------------------------------------------------------------------

feature_lookup = {
    image_id: idx
    for idx, image_id in enumerate(yolo26_image_ids)
}

predictions = {}
generation_start = time.time()

for image_number, image_id in enumerate(test_image_ids):

    feature_idx = feature_lookup[image_id]

    prediction = generate_caption_greedy(
        model,
        torch.from_numpy(
            yolo26_object_ids[feature_idx]
        ).long(),
        torch.from_numpy(
            yolo26_spatial[feature_idx]
        ).float(),
        torch.from_numpy(
            yolo26_confidence[feature_idx]
        ).float(),
        torch.from_numpy(
            yolo26_object_mask[feature_idx]
        ).bool()
    )

    predictions[image_id] = prediction

generation_time = time.time() - generation_start

print("\nGeneration complete:")
print(f"  Images: {len(predictions):,}")
print(f"  Time:   {generation_time / 60:.2f} min")
print(
    f"  Speed:  "
    f"{len(predictions) / generation_time:.2f} images/sec"
)

# ---------------------------------------------------------------------------
# Save predictions permanently
# ---------------------------------------------------------------------------

EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

prediction_file = (
    EVAL_DIR /
    "yolo26_transformer_test_predictions.json"
)

with open(
    prediction_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        predictions,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved permanently:")
print(prediction_file)

# ---------------------------------------------------------------------------
# Show a few examples ONLY
# ---------------------------------------------------------------------------

print("\nSample predictions:")

for image_id in test_image_ids[:5]:

    print(f"\n{image_id}")
    print("  →", predictions[image_id])

print("\n✓ Best model evaluated")
print("✓ Test captions generated")
print("✓ Predictions saved to Drive")
print("✓ No batch output")

print("=" * 75)
print("CELL 8 COMPLETE")
print("=" * 75)

CELL 8 — YOLO26 BEST MODEL + TEST CAPTION GENERATION
✓ Best checkpoint loaded
  Epoch: 20
  Best validation loss: 3.629171

Special tokens:
  PAD:   0
  START: 1
  END:   2
  UNK:   3

Test images: 3988

Generation complete:
  Images: 3,988
  Time:   4.77 min
  Speed:  13.92 images/sec


NameError: name 'EVAL_DIR' is not defined

In [ ]:
# ===========================================================================
# CELL 8 FIX — SAVE GENERATED TEST PREDICTIONS
# ===========================================================================

EVAL_DIR = (
    OUTPUT_DIR /
    "Evaluation"
)

EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

prediction_file = (
    EVAL_DIR /
    "yolo26_transformer_test_predictions.json"
)

with open(
    prediction_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        predictions,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 75)
print("CELL 8 — PREDICTIONS SAVED")
print("=" * 75)

print(f"Test images: {len(predictions):,}")
print(f"File: {prediction_file}")

print("\n✓ 3,988 predictions preserved")
print("✓ Best Epoch-20 checkpoint used")
print("✓ Predictions saved permanently to Drive")
print("=" * 75)
print("CELL 8 COMPLETE")
print("=" * 75)

CELL 8 — PREDICTIONS SAVED
Test images: 3,988
File: /content/drive/MyDrive/Image_captioning/Outputs/YOLO26_Object_Aware_Transformer/Evaluation/yolo26_transformer_test_predictions.json

✓ 3,988 predictions preserved
✓ Best Epoch-20 checkpoint used
✓ Predictions saved permanently to Drive
CELL 8 COMPLETE


In [ ]:
# ===========================================================================
# CELL 9 — YOLO26 TEST EVALUATION
# ===========================================================================

print("=" * 75)
print("CELL 9 — YOLO26 TEST EVALUATION")
print("=" * 75)

# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

from collections import defaultdict
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import re
import json
import math
import numpy as np

# ---------------------------------------------------------------------------
# Load predictions from Drive
# ---------------------------------------------------------------------------

prediction_file = (
    OUTPUT_DIR /
    "Evaluation" /
    "yolo26_transformer_test_predictions.json"
)

assert prediction_file.exists(), (
    f"Prediction file not found:\n{prediction_file}"
)

with open(
    prediction_file,
    "r",
    encoding="utf-8"
) as f:

    predictions = json.load(f)

predictions = {
    str(k): str(v)
    for k, v in predictions.items()
}

print("Predictions loaded:", f"{len(predictions):,}")

assert len(predictions) == 3988

# ===========================================================================
# LOAD TEST REFERENCES
# ===========================================================================

caption_npz = np.load(
    CAPTION_SEQ_FILE,
    allow_pickle=True
)

reference_image_ids = (
    caption_npz["image_ids"]
    .astype(str)
)

reference_caption_ids = (
    caption_npz["caption_ids"]
    .astype(np.int64)
)

# ---------------------------------------------------------------------------
# Build image → reference captions
# ---------------------------------------------------------------------------

references_by_image = defaultdict(list)

for image_id, caption_ids in zip(
    reference_image_ids,
    reference_caption_ids
):

    if image_id not in set(test_image_ids):
        continue

    words = []

    for token_id in caption_ids:

        token_id = int(token_id)

        if token_id in (
            PAD_ID,
            START_ID,
            END_ID
        ):
            continue

        word = id_to_caption.get(
            token_id,
            "<unk>"
        )

        words.append(word)

    if words:
        references_by_image[image_id].append(words)

# ---------------------------------------------------------------------------
# Verify reference structure
# ---------------------------------------------------------------------------

test_reference_counts = [
    len(references_by_image[image_id])
    for image_id in predictions
]

print(
    "Reference captions:",
    f"{sum(test_reference_counts):,}"
)

print(
    "Average references/image:",
    f"{np.mean(test_reference_counts):.2f}"
)

assert all(
    count == 5
    for count in test_reference_counts
), "Expected exactly 5 references per test image."

assert sum(test_reference_counts) == 19940

print("✓ 3,988 test images")
print("✓ 19,940 reference captions")
print("✓ 5 references per image")

# ===========================================================================
# TOKENIZATION
# ===========================================================================

def tokenize(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        "",
        text
    )

    return text.split()


# ---------------------------------------------------------------------------
# Prepare evaluation data
# ---------------------------------------------------------------------------

image_order = sorted(predictions.keys())

candidate_tokens = []
reference_tokens = []

for image_id in image_order:

    candidate = tokenize(
        predictions[image_id]
    )

    refs = [
        tokenize(" ".join(ref))
        for ref in references_by_image[image_id]
    ]

    candidate_tokens.append(candidate)
    reference_tokens.append(refs)

# ===========================================================================
# BLEU
# ===========================================================================

smoothing = SmoothingFunction().method1

bleu1 = corpus_bleu(
    reference_tokens,
    candidate_tokens,
    weights=(1.0, 0, 0, 0),
    smoothing_function=smoothing
)

bleu2 = corpus_bleu(
    reference_tokens,
    candidate_tokens,
    weights=(0.5, 0.5, 0, 0),
    smoothing_function=smoothing
)

bleu3 = corpus_bleu(
    reference_tokens,
    candidate_tokens,
    weights=(1/3, 1/3, 1/3, 0),
    smoothing_function=smoothing
)

bleu4 = corpus_bleu(
    reference_tokens,
    candidate_tokens,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=smoothing
)

print("\nBLEU:")
print(f"  BLEU-1: {bleu1:.6f}")
print(f"  BLEU-2: {bleu2:.6f}")
print(f"  BLEU-3: {bleu3:.6f}")
print(f"  BLEU-4: {bleu4:.6f}")

# ===========================================================================
# METEOR
# ===========================================================================

meteor_scores = []

for candidate, refs in zip(
    candidate_tokens,
    reference_tokens
):

    if not candidate:
        meteor_scores.append(0.0)
        continue

    scores = []

    for ref in refs:

        if not ref:
            scores.append(0.0)
            continue

        try:

            score = meteor_score(
                [ref],
                candidate
            )

        except Exception:

            score = 0.0

        scores.append(score)

    meteor_scores.append(
        max(scores) if scores else 0.0
    )

meteor = float(
    np.mean(meteor_scores)
)

print("\nMETEOR:")
print(f"  METEOR: {meteor:.6f}")

# ===========================================================================
# ROUGE-L
# ===========================================================================

def lcs_length(a, b):

    if not a or not b:
        return 0

    previous = [0] * (len(b) + 1)

    for token_a in a:

        current = [0]

        for j, token_b in enumerate(b, start=1):

            if token_a == token_b:

                current.append(
                    previous[j - 1] + 1
                )

            else:

                current.append(
                    max(
                        previous[j],
                        current[-1]
                    )
                )

        previous = current

    return previous[-1]


def rouge_l_score(candidate, reference):

    if not candidate or not reference:
        return 0.0

    lcs = lcs_length(
        candidate,
        reference
    )

    precision = lcs / len(candidate)
    recall = lcs / len(reference)

    if precision + recall == 0:
        return 0.0

    beta = 1.2

    return (
        (1 + beta ** 2)
        * precision
        * recall
        / (
            recall
            + beta ** 2 * precision
        )
    )


rouge_scores = []

for candidate, refs in zip(
    candidate_tokens,
    reference_tokens
):

    scores = [
        rouge_l_score(
            candidate,
            ref
        )
        for ref in refs
    ]

    rouge_scores.append(
        max(scores) if scores else 0.0
    )

rouge_l = float(
    np.mean(rouge_scores)
)

print("\nROUGE-L:")
print(f"  ROUGE-L: {rouge_l:.6f}")

# ===========================================================================
# CIDEr
# ===========================================================================

def ngrams(tokens, n):

    return [
        tuple(tokens[i:i+n])
        for i in range(
            len(tokens) - n + 1
        )
    ]


def cider_score(
    candidate,
    references
):

    if not candidate:
        return 0.0

    scores = []

    for n in range(1, 5):

        candidate_ngrams = ngrams(
            candidate,
            n
        )

        if not candidate_ngrams:
            continue

        candidate_counts = defaultdict(int)

        for gram in candidate_ngrams:
            candidate_counts[gram] += 1

        reference_counts = []

        for reference in references:

            counts = defaultdict(int)

            for gram in ngrams(
                reference,
                n
            ):
                counts[gram] += 1

            reference_counts.append(
                counts
            )

        # Document frequency
        document_frequency = defaultdict(int)

        for counts in reference_counts:

            for gram in counts:
                document_frequency[gram] += 1

        num_refs = len(
            reference_counts
        )

        # TF-IDF candidate vector
        candidate_vector = {}

        for gram, count in candidate_counts.items():

            tf = count / len(
                candidate_ngrams
            )

            df = document_frequency.get(
                gram,
                0
            )

            idf = math.log(
                (num_refs + 1)
                / (df + 1)
            )

            candidate_vector[gram] = tf * idf

        # Compare against each reference
        reference_similarities = []

        for counts in reference_counts:

            reference_ngrams_count = sum(
                counts.values()
            )

            if reference_ngrams_count == 0:
                continue

            reference_vector = {}

            for gram, count in counts.items():

                tf = (
                    count
                    / reference_ngrams_count
                )

                df = document_frequency.get(
                    gram,
                    0
                )

                idf = math.log(
                    (num_refs + 1)
                    / (df + 1)
                )

                reference_vector[gram] = (
                    tf * idf
                )

            common = set(
                candidate_vector
            ) & set(
                reference_vector
            )

            numerator = sum(
                candidate_vector[g]
                * reference_vector[g]
                for g in common
            )

            candidate_norm = math.sqrt(
                sum(
                    v * v
                    for v in candidate_vector.values()
                )
            )

            reference_norm = math.sqrt(
                sum(
                    v * v
                    for v in reference_vector.values()
                )
            )

            if (
                candidate_norm == 0
                or reference_norm == 0
            ):
                similarity = 0.0

            else:

                similarity = (
                    numerator
                    / (
                        candidate_norm
                        * reference_norm
                    )
                )

            reference_similarities.append(
                similarity
            )

        if reference_similarities:

            scores.append(
                max(reference_similarities)
            )

    if not scores:
        return 0.0

    return 10.0 * float(
        np.mean(scores)
    )


cider_scores = []

for candidate, refs in zip(
    candidate_tokens,
    reference_tokens
):

    cider_scores.append(
        cider_score(
            candidate,
            refs
        )
    )

cider = float(
    np.mean(cider_scores)
)

print("\nCIDEr:")
print(f"  CIDEr: {cider:.6f}")

# ===========================================================================
# FINAL METRICS
# ===========================================================================

metrics = {
    "model": "YOLO26s Object-Aware Transformer",
    "checkpoint_epoch": int(
        checkpoint["epoch"]
    ) if "checkpoint" in globals()
    else 20,

    "test_images": 3988,
    "reference_captions": 19940,
    "references_per_image": 5,

    "BLEU-1": float(bleu1),
    "BLEU-2": float(bleu2),
    "BLEU-3": float(bleu3),
    "BLEU-4": float(bleu4),
    "METEOR": float(meteor),
    "ROUGE-L": float(rouge_l),
    "CIDEr": float(cider),

    "SPICE": None,
    "SPICE_status": "Unavailable"
}

metrics_file = (
    EVAL_DIR /
    "yolo26_transformer_final_metrics.json"
)

with open(
    metrics_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metrics,
        f,
        indent=2
    )

print("\n" + "=" * 75)
print("🔥 YOLO26 EVALUATION COMPLETE")
print("=" * 75)

print(f"BLEU-1:  {bleu1:.6f}")
print(f"BLEU-2:  {bleu2:.6f}")
print(f"BLEU-3:  {bleu3:.6f}")
print(f"BLEU-4:  {bleu4:.6f}")
print(f"METEOR:  {meteor:.6f}")
print(f"ROUGE-L: {rouge_l:.6f}")
print(f"CIDEr:   {cider:.6f}")
print("SPICE:   Unavailable")

print("\nSaved permanently:")
print(metrics_file)

print("\n✓ Same test split")
print("✓ 5 references per image")
print("✓ Metrics saved to Drive")
print("=" * 75)
print("CELL 9 COMPLETE")
print("=" * 75)

CELL 9 — YOLO26 TEST EVALUATION
Predictions loaded: 3,988
Reference captions: 19,940
Average references/image: 5.00
✓ 3,988 test images
✓ 19,940 reference captions
✓ 5 references per image

BLEU:
  BLEU-1: 0.582644
  BLEU-2: 0.389565
  BLEU-3: 0.257530
  BLEU-4: 0.172377

METEOR:
  METEOR: 0.006534

ROUGE-L:
  ROUGE-L: 0.407225

CIDEr:
  CIDEr: 1.032395

🔥 YOLO26 EVALUATION COMPLETE
BLEU-1:  0.582644
BLEU-2:  0.389565
BLEU-3:  0.257530
BLEU-4:  0.172377
METEOR:  0.006534
ROUGE-L: 0.407225
CIDEr:   1.032395
SPICE:   Unavailable

Saved permanently:
/content/drive/MyDrive/Image_captioning/Outputs/YOLO26_Object_Aware_Transformer/Evaluation/yolo26_transformer_final_metrics.json

✓ Same test split
✓ 5 references per image
✓ Metrics saved to Drive
CELL 9 COMPLETE


In [ ]:
# ===========================================================================
# CELL 10 — YOLO26 METRIC VALIDATION / CORRECTION
# ===========================================================================

print("=" * 75)
print("CELL 10 — YOLO26 METRIC VALIDATION")
print("=" * 75)

# ---------------------------------------------------------------------------
# Required imports
# ---------------------------------------------------------------------------

import nltk
from nltk.translate.meteor_score import meteor_score
from collections import defaultdict
import numpy as np
import json
import re
import math

# ---------------------------------------------------------------------------
# Ensure WordNet is available for NLTK METEOR
# ---------------------------------------------------------------------------

try:
    from nltk.corpus import wordnet

    wordnet.synsets("dog")

except LookupError:

    print("Downloading WordNet...")

    nltk.download(
        "wordnet",
        quiet=True
    )

    nltk.download(
        "omw-1.4",
        quiet=True
    )

print("✓ WordNet available")

# ===========================================================================
# LOAD PREDICTIONS
# ===========================================================================

prediction_file = (
    OUTPUT_DIR /
    "Evaluation" /
    "yolo26_transformer_test_predictions.json"
)

with open(
    prediction_file,
    "r",
    encoding="utf-8"
) as f:

    predictions = json.load(f)

predictions = {
    str(k): str(v)
    for k, v in predictions.items()
}

# ===========================================================================
# LOAD TEST REFERENCES
# ===========================================================================

caption_npz = np.load(
    CAPTION_SEQ_FILE,
    allow_pickle=True
)

reference_image_ids = (
    caption_npz["image_ids"]
    .astype(str)
)

reference_caption_ids = (
    caption_npz["caption_ids"]
    .astype(np.int64)
)

# ---------------------------------------------------------------------------
# Test image set
# ---------------------------------------------------------------------------

test_image_set = set(
    test_image_ids
)

# ---------------------------------------------------------------------------
# Build references
# ---------------------------------------------------------------------------

references_by_image = defaultdict(list)

for image_id, caption_ids in zip(
    reference_image_ids,
    reference_caption_ids
):

    if image_id not in test_image_set:
        continue

    words = []

    for token_id in caption_ids:

        token_id = int(token_id)

        if token_id in (
            PAD_ID,
            START_ID,
            END_ID
        ):
            continue

        word = id_to_caption.get(
            token_id,
            "<unk>"
        )

        words.append(word)

    if words:
        references_by_image[image_id].append(
            words
        )

# ===========================================================================
# TOKENIZATION
# ===========================================================================

def tokenize_eval(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        "",
        text
    )

    return text.split()


# ===========================================================================
# PREPARE EVALUATION SET
# ===========================================================================

image_order = sorted(
    predictions.keys()
)

candidate_tokens = []
reference_tokens = []

for image_id in image_order:

    candidate = tokenize_eval(
        predictions[image_id]
    )

    refs = [
        tokenize_eval(
            " ".join(ref)
        )
        for ref in references_by_image[image_id]
    ]

    candidate_tokens.append(
        candidate
    )

    reference_tokens.append(
        refs
    )

# ===========================================================================
# METEOR — VERIFIED IMPLEMENTATION
# ===========================================================================

meteor_scores = []

for candidate, refs in zip(
    candidate_tokens,
    reference_tokens
):

    if not candidate:

        meteor_scores.append(0.0)
        continue

    valid_refs = [
        ref
        for ref in refs
        if ref
    ]

    if not valid_refs:

        meteor_scores.append(0.0)
        continue

    # NLTK METEOR expects:
    # meteor_score(references, hypothesis)

    score = meteor_score(
        valid_refs,
        candidate
    )

    meteor_scores.append(
        float(score)
    )

meteor_verified = float(
    np.mean(meteor_scores)
)

print("\nMETEOR VALIDATION")
print("-" * 50)
print(
    f"Images evaluated: {len(meteor_scores):,}"
)
print(
    f"METEOR:           {meteor_verified:.6f}"
)

# ===========================================================================
# BASIC SANITY CHECK
# ===========================================================================

nonempty_predictions = sum(
    bool(candidate)
    for candidate in candidate_tokens
)

unk_predictions = sum(
    "<unk>" in candidate
    for candidate in candidate_tokens
)

avg_prediction_length = np.mean(
    [
        len(candidate)
        for candidate in candidate_tokens
    ]
)

print("\nPrediction sanity:")
print(
    f"  Non-empty predictions: "
    f"{nonempty_predictions:,}/{len(candidate_tokens):,}"
)

print(
    f"  Predictions containing <unk>: "
    f"{unk_predictions:,}"
)

print(
    f"  Average token length: "
    f"{avg_prediction_length:.2f}"
)

# ===========================================================================
# SHOW A FEW EXAMPLES
# ===========================================================================

print("\nSample evaluation pairs:")

for i in range(min(3, len(image_order))):

    image_id = image_order[i]

    print("\n" + image_id)

    print(
        "Prediction:",
        predictions[image_id]
    )

    print(
        "Reference 1:",
        " ".join(
            references_by_image[image_id][0]
        )
    )

# ===========================================================================
# SAVE CORRECTED METEOR
# ===========================================================================

corrected_metrics_file = (
    EVAL_DIR /
    "yolo26_transformer_meteor_verified.json"
)

meteor_result = {
    "model":
        "YOLO26s Object-Aware Transformer",

    "test_images":
        len(image_order),

    "METEOR":
        meteor_verified,

    "evaluation":
        "NLTK METEOR with WordNet",

    "wordnet":
        True
}

with open(
    corrected_metrics_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        meteor_result,
        f,
        indent=2
    )

print("\nSaved permanently:")
print(corrected_metrics_file)

print("\n" + "=" * 75)
print("CELL 10 COMPLETE")
print("=" * 75)

CELL 10 — YOLO26 METRIC VALIDATION
✓ WordNet available

METEOR VALIDATION
--------------------------------------------------
Images evaluated: 3,988
METEOR:           0.353311

Prediction sanity:
  Non-empty predictions: 3,988/3,988
  Predictions containing <unk>: 0
  Average token length: 10.95

Sample evaluation pairs:

Flickr30K/10002456.jpg
Prediction: a man in a blue shirt is standing in front of a large window .
Reference 1: several men in hard hats are operating a giant pulley system .

Flickr30K/1000523639.jpg
Prediction: a man and a woman are dancing in a studio .
Reference 1: two people in the photo are playing the guitar and the other is poking at him .

Flickr30K/1003163366.jpg
Prediction: a man in a blue shirt and jeans is sitting on a bench with a dog in the background .
Reference 1: a man sleeping on a bench outside with a white and black dog sitting next to him .

Saved permanently:
/content/drive/MyDrive/Image_captioning/Outputs/YOLO26_Object_Aware_Transformer/Evaluati

In [ ]:
# ===========================================================================
# CELL 11 — YOLO26 STANDARD CIDEr — CORRECTED
# ===========================================================================

print("=" * 75)
print("CELL 11 — YOLO26 STANDARD CIDEr EVALUATION")
print("=" * 75)

from pycocoevalcap.cider.cider import Cider
import numpy as np
import json

# ---------------------------------------------------------------------------
# Load predictions
# ---------------------------------------------------------------------------

prediction_file = (
    OUTPUT_DIR /
    "Evaluation" /
    "yolo26_transformer_test_predictions.json"
)

assert prediction_file.exists(), (
    f"Prediction file missing:\n{prediction_file}"
)

with open(
    prediction_file,
    "r",
    encoding="utf-8"
) as f:

    predictions = json.load(f)

predictions = {
    str(k): str(v)
    for k, v in predictions.items()
}

# ===========================================================================
# LOAD REFERENCES
# ===========================================================================

caption_npz = np.load(
    CAPTION_SEQ_FILE,
    allow_pickle=True
)

reference_image_ids = (
    caption_npz["image_ids"]
    .astype(str)
)

reference_caption_ids = (
    caption_npz["caption_ids"]
    .astype(np.int64)
)

test_image_set = set(
    test_image_ids
)

# ---------------------------------------------------------------------------
# Build references as STRINGS
# ---------------------------------------------------------------------------

references = {}

for image_id, caption_ids in zip(
    reference_image_ids,
    reference_caption_ids
):

    if image_id not in test_image_set:
        continue

    words = []

    for token_id in caption_ids:

        token_id = int(token_id)

        if token_id in (
            PAD_ID,
            START_ID,
            END_ID
        ):
            continue

        words.append(
            id_to_caption.get(
                token_id,
                "<unk>"
            )
        )

    caption = " ".join(words)

    references.setdefault(
        image_id,
        []
    ).append(caption)

# ---------------------------------------------------------------------------
# Verify
# ---------------------------------------------------------------------------

assert len(predictions) == 3988
assert len(references) == 3988

assert all(
    len(references[image_id]) == 5
    for image_id in predictions
)

print("Predictions:", f"{len(predictions):,}")
print("Reference images:", f"{len(references):,}")
print("References/image: 5")

# ===========================================================================
# BUILD pycocoevalcap FORMAT
#
# IMPORTANT:
# gts[id] = list of STRING captions
# res[id] = list containing ONE STRING caption
# ===========================================================================

gts = {}
res = {}

image_order = sorted(
    predictions.keys()
)

for numeric_id, image_id in enumerate(
    image_order
):

    gts[numeric_id] = references[image_id]

    res[numeric_id] = [
        predictions[image_id]
    ]

# ===========================================================================
# STANDARD CIDEr
# ===========================================================================

print("\nCalculating standard CIDEr...")
print("No caption regeneration required.")

cider_scorer = Cider()

cider_score, cider_scores = (
    cider_scorer.compute_score(
        gts,
        res
    )
)

cider_score = float(cider_score)

print("\n" + "=" * 75)
print("STANDARD CIDEr RESULT")
print("=" * 75)

print(
    f"CIDEr: {cider_score:.6f}"
)

# ===========================================================================
# SAVE
# ===========================================================================

cider_file = (
    EVAL_DIR /
    "yolo26_standard_cider.json"
)

cider_result = {
    "model":
        "YOLO26s Object-Aware Transformer",

    "test_images":
        3988,

    "reference_captions":
        19940,

    "references_per_image":
        5,

    "CIDEr":
        cider_score,

    "implementation":
        "pycocoevalcap CIDEr"
}

with open(
    cider_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        cider_result,
        f,
        indent=2
    )

print("\nSaved permanently:")
print(cider_file)

print("\n✓ Standard CIDEr calculated")
print("✓ Same 3,988-image test set")
print("✓ Same 19,940 reference captions")
print("✓ Result saved to Drive")

print("=" * 75)
print("CELL 11 COMPLETE")
print("=" * 75)

CELL 11 — YOLO26 STANDARD CIDEr EVALUATION
Predictions: 3,988
Reference images: 3,988
References/image: 5

Calculating standard CIDEr...
No caption regeneration required.

STANDARD CIDEr RESULT
CIDEr: 0.336691

Saved permanently:
/content/drive/MyDrive/Image_captioning/Outputs/YOLO26_Object_Aware_Transformer/Evaluation/yolo26_standard_cider.json

✓ Standard CIDEr calculated
✓ Same 3,988-image test set
✓ Same 19,940 reference captions
✓ Result saved to Drive
CELL 11 COMPLETE


In [ ]:
# ============================================================================
# NOTEBOOK 13 — CELL A
# COMPLETE YOLO26 EVALUATION RECOVERY
# ============================================================================
print("=" * 80)
print("CELL A — COMPLETE YOLO26 EVALUATION RECOVERY")
print("=" * 80)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from collections import defaultdict
import json
import re
import numpy as np
import pandas as pd

# ============================================================================
# PERMANENT DRIVE PATHS
# ============================================================================

DRIVE_ROOT = Path("/content/drive/MyDrive/Image_captioning")

OUTPUT_DIR = (
    DRIVE_ROOT /
    "Outputs" /
    "YOLO26_Object_Aware_Transformer"
)

EVAL_DIR = OUTPUT_DIR / "Evaluation"

CAPTION_VOCAB_FILE = (
    DRIVE_ROOT /
    "Outputs" /
    "Structured_Captioning" /
    "caption_vocabulary_train_only.json"
)

CAPTION_SEQ_FILE = (
    DRIVE_ROOT /
    "Features" /
    "Structured_Object" /
    "caption_sequences_train_only.npz"
)

TEST_SPLIT_FILE = (
    DRIVE_ROOT /
    "Outputs" /
    "Structured_Captioning" /
    "test_image_ids.csv"
)

PREDICTION_FILE = (
    EVAL_DIR /
    "yolo26_transformer_test_predictions.json"
)

METEOR_FILE = (
    EVAL_DIR /
    "yolo26_transformer_meteor_verified.json"
)

CIDER_FILE = (
    EVAL_DIR /
    "yolo26_standard_cider.json"
)

# ============================================================================
# VERIFY PERMANENT FILES
# ============================================================================

required_files = {
    "Caption vocabulary": CAPTION_VOCAB_FILE,
    "Caption sequences": CAPTION_SEQ_FILE,
    "Test split": TEST_SPLIT_FILE,
    "YOLO26 predictions": PREDICTION_FILE,
}

print("\nChecking permanent Drive files...")

for name, path in required_files.items():
    print(f"{name:25s}: ", end="")
    assert path.exists(), f"MISSING: {path}"
    print("✓")

# ============================================================================
# CAPTION VOCABULARY
# ============================================================================

with open(
    CAPTION_VOCAB_FILE,
    "r",
    encoding="utf-8"
) as f:
    vocab_raw = json.load(f)

caption_to_id = vocab_raw.get(
    "caption_to_id",
    vocab_raw
)

caption_to_id = {
    str(k): int(v)
    for k, v in caption_to_id.items()
}

id_to_caption = {
    int(v): str(k)
    for k, v in caption_to_id.items()
}

PAD_ID = caption_to_id["<pad>"]
START_ID = caption_to_id["<start>"]
END_ID = caption_to_id["<end>"]
UNK_ID = caption_to_id["<unk>"]

assert len(caption_to_id) == 12666
assert PAD_ID == 0
assert START_ID == 1
assert END_ID == 2
assert UNK_ID == 3

# ============================================================================
# CAPTION SEQUENCES
# ============================================================================

caption_data = np.load(
    CAPTION_SEQ_FILE,
    allow_pickle=True
)

caption_image_ids = (
    caption_data["image_ids"]
    .astype(str)
)

caption_sequences = (
    caption_data["caption_ids"]
    .astype(np.int64)
)

assert caption_image_ids.shape == (199369,)
assert caption_sequences.shape == (199369, 32)

# ============================================================================
# TEST SPLIT
# ============================================================================

test_df = pd.read_csv(TEST_SPLIT_FILE)

test_column = (
    "image_id"
    if "image_id" in test_df.columns
    else test_df.columns[0]
)

test_image_ids = (
    test_df[test_column]
    .astype(str)
    .str.strip()
    .tolist()
)

test_image_set = set(test_image_ids)

assert len(test_image_ids) == 3988

# ============================================================================
# LOAD PERMANENT PREDICTIONS
# ============================================================================

with open(
    PREDICTION_FILE,
    "r",
    encoding="utf-8"
) as f:
    predictions = json.load(f)

predictions = {
    str(k): str(v)
    for k, v in predictions.items()
}

assert len(predictions) == 3988
assert set(predictions.keys()) == test_image_set

# ============================================================================
# BUILD TEST REFERENCES
# ============================================================================

references_by_image = defaultdict(list)

for image_id, token_ids in zip(
    caption_image_ids,
    caption_sequences
):

    if image_id not in test_image_set:
        continue

    words = []

    for token_id in token_ids:

        token_id = int(token_id)

        if token_id == PAD_ID:
            continue

        if token_id == START_ID:
            continue

        if token_id == END_ID:
            break

        words.append(
            id_to_caption.get(
                token_id,
                "<unk>"
            )
        )

    caption = " ".join(words).strip()

    if caption:
        references_by_image[image_id].append(caption)

# ============================================================================
# FINAL STRUCTURAL CHECKS
# ============================================================================

assert len(references_by_image) == 3988

reference_counts = [
    len(refs)
    for refs in references_by_image.values()
]

assert min(reference_counts) == 5
assert max(reference_counts) == 5

assert all(
    image_id in references_by_image
    for image_id in predictions
)

# ============================================================================
# TOKENIZATION
# ============================================================================

def tokenize_eval(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        "",
        text
    )

    return text.split()

# ============================================================================
# PREPARE METRIC DATA
# ============================================================================

image_order = sorted(predictions.keys())

candidate_tokens = [
    tokenize_eval(predictions[image_id])
    for image_id in image_order
]

reference_tokens = [
    [
        tokenize_eval(reference)
        for reference in references_by_image[image_id]
    ]
    for image_id in image_order
]

# ============================================================================
# SANITY
# ============================================================================

empty_predictions = sum(
    1
    for candidate in candidate_tokens
    if not candidate
)

unk_predictions = sum(
    1
    for candidate in candidate_tokens
    if "<unk>" in candidate
)

average_length = np.mean([
    len(candidate)
    for candidate in candidate_tokens
])

print("\n" + "=" * 80)
print("RECOVERY COMPLETE")
print("=" * 80)

print(f"Test images:             {len(predictions):,}")
print(f"Reference captions:      {sum(reference_counts):,}")
print(f"References/image:        {np.mean(reference_counts):.2f}")
print(f"Caption vocabulary:      {len(caption_to_id):,}")
print(f"Empty predictions:       {empty_predictions:,}")
print(f"Predictions with <unk>:  {unk_predictions:,}")
print(f"Average prediction len:  {average_length:.2f}")

assert empty_predictions == 0
assert unk_predictions == 0

print("\n✓ Predictions recovered from Drive")
print("✓ References rebuilt from permanent caption sequences")
print("✓ Exact 3,988-image test split restored")
print("✓ 5 references per image restored")
print("✓ No model loading required")
print("✓ No training required")
print("✓ No YOLO inference required")

print("=" * 80)
print("CELL A COMPLETE")
print("=" * 80)

CELL A — COMPLETE YOLO26 EVALUATION RECOVERY
Mounted at /content/drive

Checking permanent Drive files...
Caption vocabulary       : ✓
Caption sequences        : ✓
Test split               : ✓
YOLO26 predictions       : ✓

RECOVERY COMPLETE
Test images:             3,988
Reference captions:      19,940
References/image:        5.00
Caption vocabulary:      12,666
Empty predictions:       0
Predictions with <unk>:  0
Average prediction len:  10.95

✓ Predictions recovered from Drive
✓ References rebuilt from permanent caption sequences
✓ Exact 3,988-image test split restored
✓ 5 references per image restored
✓ No model loading required
✓ No training required
✓ No YOLO inference required
CELL A COMPLETE


In [ ]:
# ============================================================================
# NOTEBOOK 13 — CELL B
# FINAL YOLO26 METRICS
# ============================================================================
print("=" * 80)
print("CELL B — FINAL YOLO26 METRICS")
print("=" * 80)

import json
import numpy as np

from nltk.translate.bleu_score import (
    corpus_bleu,
    SmoothingFunction
)

# ============================================================================
# BLEU-1 TO BLEU-4
# ============================================================================

smoothing = SmoothingFunction().method1

bleu1 = corpus_bleu(
    reference_tokens,
    candidate_tokens,
    weights=(1.0, 0.0, 0.0, 0.0),
    smoothing_function=smoothing
)

bleu2 = corpus_bleu(
    reference_tokens,
    candidate_tokens,
    weights=(0.5, 0.5, 0.0, 0.0),
    smoothing_function=smoothing
)

bleu3 = corpus_bleu(
    reference_tokens,
    candidate_tokens,
    weights=(1/3, 1/3, 1/3, 0.0),
    smoothing_function=smoothing
)

bleu4 = corpus_bleu(
    reference_tokens,
    candidate_tokens,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=smoothing
)

# ============================================================================
# ROUGE-L
# ============================================================================

def lcs_length(a, b):

    if not a or not b:
        return 0

    previous = [0] * (len(b) + 1)

    for token_a in a:

        current = [0]

        for j, token_b in enumerate(b, start=1):

            if token_a == token_b:
                current.append(
                    previous[j - 1] + 1
                )

            else:
                current.append(
                    max(
                        previous[j],
                        current[-1]
                    )
                )

        previous = current

    return previous[-1]


def rouge_l_score(candidate, reference):

    if not candidate or not reference:
        return 0.0

    lcs = lcs_length(
        candidate,
        reference
    )

    precision = lcs / len(candidate)
    recall = lcs / len(reference)

    if precision + recall == 0:
        return 0.0

    beta = 1.2

    return (
        (1 + beta ** 2)
        * precision
        * recall
        / (
            recall
            + beta ** 2 * precision
        )
    )


rouge_scores = []

for candidate, refs in zip(
    candidate_tokens,
    reference_tokens
):

    scores = [
        rouge_l_score(
            candidate,
            reference
        )
        for reference in refs
    ]

    rouge_scores.append(
        max(scores)
        if scores
        else 0.0
    )

rouge_l = float(
    np.mean(rouge_scores)
)

# ============================================================================
# VERIFIED METEOR
# ============================================================================
#
# The notebook already calculated and permanently saved:
# NLTK METEOR + WordNet = 0.353311
#
# We load that verified result instead of accidentally reproducing
# the earlier incorrect 0.006534 result.
# ============================================================================

meteor = None

if METEOR_FILE.exists():

    with open(
        METEOR_FILE,
        "r",
        encoding="utf-8"
    ) as f:
        meteor_data = json.load(f)

    meteor = float(
        meteor_data["METEOR"]
    )

    meteor_implementation = (
        meteor_data.get(
            "evaluation",
            "NLTK METEOR with WordNet"
        )
    )

else:

    # Fallback: calculate it now.
    import nltk
    from nltk.translate.meteor_score import meteor_score

    try:
        from nltk.corpus import wordnet
        wordnet.synsets("dog")
    except LookupError:
        nltk.download(
            "wordnet",
            quiet=True
        )
        nltk.download(
            "omw-1.4",
            quiet=True
        )

    meteor_scores = []

    for candidate, refs in zip(
        candidate_tokens,
        reference_tokens
    ):

        valid_refs = [
            ref
            for ref in refs
            if ref
        ]

        if not candidate or not valid_refs:
            meteor_scores.append(0.0)
            continue

        meteor_scores.append(
            float(
                meteor_score(
                    valid_refs,
                    candidate
                )
            )
        )

    meteor = float(
        np.mean(meteor_scores)
    )

    meteor_implementation = (
        "NLTK METEOR with WordNet"
    )

# ============================================================================
# STANDARD CIDEr
# ============================================================================
#
# The notebook separately calculated standard pycocoevalcap CIDEr.
# Load that permanent result.
# ============================================================================

cider = None
cider_implementation = None

if CIDER_FILE.exists():

    with open(
        CIDER_FILE,
        "r",
        encoding="utf-8"
    ) as f:
        cider_data = json.load(f)

    cider = float(
        cider_data["CIDEr"]
    )

    cider_implementation = (
        cider_data.get(
            "implementation",
            "pycocoevalcap CIDEr"
        )
    )

else:

    # If the permanent CIDEr JSON is missing, calculate it.
    try:

        from pycocoevalcap.cider.cider import Cider

        gts = {}
        res = {}

        for numeric_id, image_id in enumerate(
            image_order
        ):

            gts[numeric_id] = (
                references_by_image[image_id]
            )

            res[numeric_id] = [
                predictions[image_id]
            ]

        cider_scorer = Cider()

        cider_score, cider_scores = (
            cider_scorer.compute_score(
                gts,
                res
            )
        )

        cider = float(
            cider_score
        )

        cider_implementation = (
            "pycocoevalcap CIDEr"
        )

    except Exception as e:

        cider = None

        cider_implementation = (
            "Unavailable in recovery runtime: "
            + str(e)
        )

# ============================================================================
# SPICE
# ============================================================================

spice = None
spice_status = "Unavailable"

# SPICE was already determined unavailable in the experiment.
# We intentionally do not rerun it.

# ============================================================================
# FINAL METRICS
# ============================================================================

final_metrics = {

    "model":
        "YOLO26s Object-Aware Transformer",

    "detector":
        "YOLO26s",

    "checkpoint_epoch":
        20,

    "test_images":
        3988,

    "reference_captions":
        19940,

    "references_per_image":
        5,

    "BLEU-1":
        float(bleu1),

    "BLEU-2":
        float(bleu2),

    "BLEU-3":
        float(bleu3),

    "BLEU-4":
        float(bleu4),

    "METEOR":
        float(meteor),

    "METEOR_implementation":
        meteor_implementation,

    "ROUGE-L":
        float(rouge_l),

    "CIDEr":
        cider,

    "CIDEr_implementation":
        cider_implementation,

    "SPICE":
        spice,

    "SPICE_status":
        spice_status
}

FINAL_METRICS_FILE = (
    EVAL_DIR /
    "yolo26_transformer_final_metrics_recovered.json"
)

with open(
    FINAL_METRICS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_metrics,
        f,
        indent=2
    )

# ============================================================================
# DISPLAY
# ============================================================================

print("\n" + "=" * 80)
print("🔥 FINAL YOLO26 TEST METRICS")
print("=" * 80)

print(f"BLEU-1 :  {bleu1:.6f}")
print(f"BLEU-2 :  {bleu2:.6f}")
print(f"BLEU-3 :  {bleu3:.6f}")
print(f"BLEU-4 :  {bleu4:.6f}")
print(f"METEOR :  {meteor:.6f}")
print(f"ROUGE-L:  {rouge_l:.6f}")

if cider is not None:
    print(f"CIDEr  :  {cider:.6f}")
else:
    print("CIDEr  :  UNAVAILABLE")

print("SPICE  :  Unavailable")

print("\n" + "-" * 80)
print("Evaluation protocol")
print("-" * 80)
print("Test images       :", f"{len(predictions):,}")
print("Reference captions:", f"{sum(reference_counts):,}")
print("References/image  :", "5")
print("Checkpoint        :", "Epoch 20")
print("Generation        :", "Greedy decoding")
print("METEOR            :", meteor_implementation)
print("CIDEr             :", cider_implementation)

print("\nSaved permanently:")
print(FINAL_METRICS_FILE)

print("\n" + "=" * 80)
print("✓ YOLO26 EVALUATION RECOVERY COMPLETE")
print("=" * 80)

CELL B — FINAL YOLO26 METRICS

🔥 FINAL YOLO26 TEST METRICS
BLEU-1 :  0.582644
BLEU-2 :  0.389565
BLEU-3 :  0.257530
BLEU-4 :  0.172377
METEOR :  0.353311
ROUGE-L:  0.407225
CIDEr  :  0.336691
SPICE  :  Unavailable

--------------------------------------------------------------------------------
Evaluation protocol
--------------------------------------------------------------------------------
Test images       : 3,988
Reference captions: 19,940
References/image  : 5
Checkpoint        : Epoch 20
Generation        : Greedy decoding
METEOR            : NLTK METEOR with WordNet
CIDEr             : pycocoevalcap CIDEr

Saved permanently:
/content/drive/MyDrive/Image_captioning/Outputs/YOLO26_Object_Aware_Transformer/Evaluation/yolo26_transformer_final_metrics_recovered.json

✓ YOLO26 EVALUATION RECOVERY COMPLETE
